# 타빌리(Tavily) 검색 도구와 LLM 도구 호출

에이전트가 학습 데이터에 없는 최신 정보를 쓰려면 **도구(tool)**가 필요하다. 타빌리는 LLM이 쓰기 좋게 결과를 정리해 주는 웹 검색 API이고, `langchain-tavily`의 `TavilySearch`로 랭체인 도구처럼 쓸 수 있다.

1. 타빌리 검색 도구를 직접 호출해서 결과 형식을 확인하고,
2. `@tool`로 만든 계산 도구와 타빌리 검색 도구를 `bind_tools()`로 LLM에 연결해, **LLM이 언제 어떤 도구를 어떤 인자로 부르겠다고 결정하는지** 확인한다.

`.env`에 `OPENAI_API_KEY`와 `TAVILY_API_KEY`가 필요하다.

## 1. 환경변수 로드

In [1]:
from dotenv import load_dotenv

# .env의 OPENAI_API_KEY, TAVILY_API_KEY를 환경변수로 불러온다.
load_dotenv()

True

## 2. 타빌리 검색 사용하기

### 2-1. 딕셔너리로 검색어 넘기기

`TavilySearch(max_results=2)`는 검색 결과를 최대 2개까지 가져오는 도구다. 도구는 `invoke()`로 실행하고, 인자는 `{"query": ...}` 딕셔너리로 넘긴다.

결과는 딕셔너리이고, `results` 안에 문서마다 `url`, `title`, `content`(본문 요약), `score`(질문과의 관련도)가 들어 있다. 웹 페이지 HTML이 아니라 LLM에 바로 넣기 좋은 텍스트로 정리돼서 온다.

In [2]:
from langchain_tavily import TavilySearch 

# 검색 결과를 최대 2개까지 가져오는 타빌리 검색 도구
tool = TavilySearch(max_results=2)
# 도구 실행: 인자를 딕셔너리로 전달
tool.invoke({"query": "랭그래프가 무엇인가요?"})

{'query': '랭그래프가 무엇인가요?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://velog.io/@ohback/LangGraph',
   'title': 'LangGraph란 무엇이고 언제 쓰면 좋을까?',
   'content': '여기서 등장하는 게 LangGraph다. 이름 그대로 그래프(노드/엣지)로 LLM 애플리케이션의 흐름을 설계한다.  \n LangChain이 블록(모델·프롬프트·툴)을 잘 조립하게 해주는 공구상자라면, LangGraph는 그 블록들이 언제, 어떤 조건으로, 어떤 상태를 들고 움직일지를 정밀하게 오케스트레이션하는 조감도 + 신호등에 가깝다.\n\n  \n\n## 그래서 LangGraph가 뭐냐면..\n\n> LangChain 에서 개발한 LangGraph는 복잡한 생성형 AI 에이전트 워크플로를 구축, 배포 및 관리하도록 설계된 오픈소스 AI 에이전트 프레임워크입니다. 사용자가 확장 가능하고 효율적인 방식으로 대규모 언어 모델 (LLM)을 생성, 실행 및 최적화할 수 있도록 지원하는 도구와 라이브러리 세트를 제공합니다. LangGraph는 그래프 기반 아키텍처의 강력한 기능을 활용하여 AI 에이전트 워크플로 의 다양한 구성 요소 간의 복잡한 관계를 모델링하고 관리합니다. [...] > LangGraph는 단일 에이전트의 한계를 뛰어넘는 다중 에이전트 시스템을 구축할 수 있게 해줍니다. 각 에이전트는 특정 목표를 향해 자율적으로 행동하며, 다른 에이전트들과 협업하여 복잡한 문제를 해결합니다. 이러한 에이전트는 복잡한 의사결정을 위한 사전 정보, 작업 단계를 단축하기 위해 도움이 되는 미리 정의된 도구들 등에 접근하여 활용할지 스스로 결정할 수 있습니다.\n\n \n\n###### 출처: \n\n  \n\n## 구성 요소 및 예시 살펴보기\n\nLangGraph는 기본적으로 에이전트 워크플로를 그래프로 모델

### 2-2. 문자열로 바로 넘기기

필수 인자가 `query` 하나뿐이라서 문자열만 넘겨도 같은 결과가 나온다. (검색 결과 `id`, `request_id`만 매번 달라진다.)

In [4]:
from langchain_tavily import TavilySearch 

tool = TavilySearch(max_results=2)
# 필수 인자가 query 하나라서 문자열만 넘겨도 된다.
tool.invoke("랭그래프가 무엇인가요?")

{'query': '랭그래프가 무엇인가요?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://velog.io/@ohback/LangGraph',
   'title': 'LangGraph란 무엇이고 언제 쓰면 좋을까?',
   'content': '여기서 등장하는 게 LangGraph다. 이름 그대로 그래프(노드/엣지)로 LLM 애플리케이션의 흐름을 설계한다.  \n LangChain이 블록(모델·프롬프트·툴)을 잘 조립하게 해주는 공구상자라면, LangGraph는 그 블록들이 언제, 어떤 조건으로, 어떤 상태를 들고 움직일지를 정밀하게 오케스트레이션하는 조감도 + 신호등에 가깝다.\n\n  \n\n## 그래서 LangGraph가 뭐냐면..\n\n> LangChain 에서 개발한 LangGraph는 복잡한 생성형 AI 에이전트 워크플로를 구축, 배포 및 관리하도록 설계된 오픈소스 AI 에이전트 프레임워크입니다. 사용자가 확장 가능하고 효율적인 방식으로 대규모 언어 모델 (LLM)을 생성, 실행 및 최적화할 수 있도록 지원하는 도구와 라이브러리 세트를 제공합니다. LangGraph는 그래프 기반 아키텍처의 강력한 기능을 활용하여 AI 에이전트 워크플로 의 다양한 구성 요소 간의 복잡한 관계를 모델링하고 관리합니다. [...] > LangGraph는 단일 에이전트의 한계를 뛰어넘는 다중 에이전트 시스템을 구축할 수 있게 해줍니다. 각 에이전트는 특정 목표를 향해 자율적으로 행동하며, 다른 에이전트들과 협업하여 복잡한 문제를 해결합니다. 이러한 에이전트는 복잡한 의사결정을 위한 사전 정보, 작업 단계를 단축하기 위해 도움이 되는 미리 정의된 도구들 등에 접근하여 활용할지 스스로 결정할 수 있습니다.\n\n \n\n###### 출처: \n\n  \n\n## 구성 요소 및 예시 살펴보기\n\nLangGraph는 기본적으로 에이전트 워크플로를 그래프로 모델

### 2-3. ToolCall 형식으로 호출하기

`{"args": ..., "type": "tool_call", "id": ..., "name": ...}`는 LLM이 "이 도구를 이 인자로 실행해 달라"고 요청할 때 쓰는 **ToolCall** 형식이다. 도구에 이 형식을 넘기면 결과를 그냥 딕셔너리가 아니라 `ToolMessage`로 감싸서 돌려준다.

In [5]:
# LLM이 만들어내는 ToolCall 형식(args, type, id, name)으로 도구를 실행
invoke_with_toolcall = tool.invoke({"args": {'query': "What's a 'node' in LangGraph?"}, "type": "tool_call", "id": "foo", "name": "tavily_search"})

`ToolMessage`의 `tool_call_id`에는 요청의 `id`(`"foo"`)가 그대로 들어간다. 에이전트는 이 id로 "어떤 도구 요청에 대한 결과인지"를 LLM에게 알려준다.

In [6]:
# 결과가 ToolMessage로 감싸져 있고, tool_call_id에 요청 id(foo)가 들어 있다.
invoke_with_toolcall

ToolMessage(content='{"query": "What\'s a \'node\' in LangGraph?", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.ibm.com/content/adobe-cms/us/en/think/topics/langgraph", "title": "What is LangGraph? | IBM", "content": "Nodes: In LangGraph, nodes represent individual components or agents within an AI workflow. Nodes can be thought of as “actors” that interact with each other in a specific way. For example, to add nodes for tool calling, one can use the ToolNode. Another example, the next node, refers to the node that will be executed following the current one.", "score": 0.926683, "raw_content": null, "id": "619e64-00"}, {"url": "https://shafiqulai.github.io/blogs/blog_8.html", "title": "LangGraph Basics: Part 1 — StateGraph, Nodes & Edges", "content": "> 💡 A node returns only the fields it updated — not the entire state. LangGraph merges the returned dictionary back into the existing state automatically. So if a node returns {\\"answer\\": 

`content`에는 검색 결과 딕셔너리가 **JSON 문자열**로 들어 있다. LLM에게 다시 넘길 메시지는 문자열이어야 하기 때문이다.

In [7]:
# 검색 결과가 JSON 문자열로 변환되어 들어 있다.
invoke_with_toolcall.content

'{"query": "What\'s a \'node\' in LangGraph?", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.ibm.com/content/adobe-cms/us/en/think/topics/langgraph", "title": "What is LangGraph? | IBM", "content": "Nodes: In LangGraph, nodes represent individual components or agents within an AI workflow. Nodes can be thought of as “actors” that interact with each other in a specific way. For example, to add nodes for tool calling, one can use the ToolNode. Another example, the next node, refers to the node that will be executed following the current one.", "score": 0.926683, "raw_content": null, "id": "619e64-00"}, {"url": "https://shafiqulai.github.io/blogs/blog_8.html", "title": "LangGraph Basics: Part 1 — StateGraph, Nodes & Edges", "content": "> 💡 A node returns only the fields it updated — not the entire state. LangGraph merges the returned dictionary back into the existing state automatically. So if a node returns {\\"answer\\": \\"...\\"}, the ques

## 3. LLM에 도구 연결하기 (bind_tools)

### 3-1. @tool로 사용자 정의 도구 만들기

`@tool` 데코레이터를 붙이면 일반 함수가 랭체인 도구가 된다. **함수 이름, docstring, 타입 힌트**가 그대로 도구 설명서(스키마)가 되어 LLM에게 전달되므로, LLM은 이걸 읽고 언제 어떤 인자로 도구를 부를지 판단한다.

In [8]:
from langchain.tools import tool 

# @tool: 함수 이름·docstring·타입 힌트가 LLM이 읽는 도구 설명서가 된다.
@tool 
def add(a: int, b: int) -> int: 
    """Adds a and b.
    
    Args:
        a: first int
        b: second int
    """
    return a + b 

@tool 
def multiply(a: int, b: int) -> int:
    """Multiplies a and b.
    
    Args:
        a: first int 
        b: second int 
    """
    return a * b 

# LLM에 연결할 도구 목록
tools = [add, multiply]

### 3-2. bind_tools로 LLM에 도구 연결

`bind_tools(tools)`는 LLM을 호출할 때마다 도구 설명서를 함께 보내도록 설정한 새 모델을 만든다.

In [9]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(model="gpt-4o")
# LLM을 호출할 때마다 도구 설명서(스키마)를 함께 보내도록 연결
llm_with_tools = llm.bind_tools(tools)

### 3-3. 도구가 필요한 질문

계산 질문을 하면 LLM은 답을 직접 쓰지 않고 `content=''`, `finish_reason='tool_calls'`로 응답한다. 대신 `tool_calls`에 **어떤 도구를 어떤 인자로 실행할지**를 담는다. 질문 하나에 계산이 두 개 들어 있어서 `multiply`와 `add` 호출 두 개를 한 번에 요청했다.

중요한 점은 `bind_tools`만으로는 **도구가 실행되지 않는다**는 것이다. LLM은 실행 계획만 세우고, 실제 실행과 결과 전달은 다음 단계(6.2.3 랭그래프 에이전트 그래프)에서 한다.

In [10]:
query = "3 곱하기 5는 뭔가요? 그리고 2 더하기 4는 뭔가요?"

# LLM이 도구가 필요하다고 판단하면 답변 대신 tool_calls를 반환한다.
response = llm_with_tools.invoke(query)
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 120, 'total_tokens': 170, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_02e43187ba', 'id': 'chatcmpl-EURkpgABYuOaMKAnzrWsuJyaSbVX1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fb76-5eb2-7f01-bcbf-ce55892983a7-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'call_9x3wHb9u2qS58fE6nmbfYveC', 'type': 'tool_call'}, {'name': 'add', 'args': {'a': 2, 'b': 4}, 'id': 'call_aXW4Ajtw8YjogdgXuRUuLAUM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 50, 'total_tokens': 170, 'input_token_details': {'a

In [11]:
# LLM이 요청한 도구 호출 목록: 도구 이름(name)과 인자(args)
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_9x3wHb9u2qS58fE6nmbfYveC',
  'type': 'tool_call'},
 {'name': 'add',
  'args': {'a': 2, 'b': 4},
  'id': 'call_aXW4Ajtw8YjogdgXuRUuLAUM',
  'type': 'tool_call'}]

### 3-4. 도구가 필요 없는 질문

인사처럼 도구가 필요 없는 질문에는 평소처럼 `content`에 답변을 쓰고 `finish_reason='stop'`으로 끝난다. 도구를 쓸지 말지도 LLM이 스스로 판단한다.

In [12]:
query = "안녕하세요."

# 도구가 필요 없으면 평소처럼 바로 답변한다.
response = llm_with_tools.invoke(query)
response

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 98, 'total_tokens': 109, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_02e43187ba', 'id': 'chatcmpl-EURogdmmkBWu2wCokXR22AEhEs0GK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0fb7a-06cb-77d3-a12b-f0addd6e1b6f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 11, 'total_tokens': 109, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
# 도구 호출 요청이 없으므로 빈 리스트
response.tool_calls

[]

## 4. 타빌리 검색 도구를 LLM에 연결하기

이번에는 계산 도구 대신 타빌리 검색 도구를 연결한다. (이 셀에서 `tool` 변수에 `TavilySearch`를 다시 담으면서, 위에서 import한 `@tool` 데코레이터 이름을 덮어쓴다. 이 노트북에서는 이후에 `@tool`을 쓰지 않아서 문제가 없지만, 변수 이름을 `search_tool`처럼 따로 두는 게 안전하다.)

In [14]:
from langchain_openai import ChatOpenAI 

# 주의: 위에서 import한 @tool 데코레이터 이름(tool)을 TavilySearch 객체로 덮어쓴다.
tool = TavilySearch(max_results=2)
# 이번에는 웹 검색 도구 하나만 연결
tools = [tool] 

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

인사에는 도구 없이 바로 답한다.

`prompt_tokens`를 보면 계산 도구를 연결했을 때(98~120)보다 10배 가까이 많은 **1189**다. 도구 설명서도 입력 토큰에 포함되는데, 타빌리 검색 도구는 설명과 선택 인자(`include_domains`, `time_range`, `topic` 등)가 많아 설명서가 길기 때문이다. 같은 버전으로 스키마를 뽑아 보면 `add`+`multiply`는 약 540자, `TavilySearch`는 약 6,500자다. 도구를 많이 연결할수록 질문과 상관없이 매 호출 비용이 늘어난다.

In [15]:
# 도구가 필요 없는 질문 -> 바로 답변 (도구 설명서 때문에 입력 토큰은 많다)
llm_with_tools.invoke("안녕")

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 1189, 'total_tokens': 1200, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_24e955c341', 'id': 'chatcmpl-EURqULa7moqEUvsJp9qB7jP0jA2m8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0fb7b-bd25-7901-a04d-ba37684ccb24-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1189, 'output_tokens': 11, 'total_tokens': 1200, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### 4-1. 최신 정보가 필요한 질문

학습 데이터에 없는 "2026년 AI 트렌드"를 물으면 LLM이 `tavily_search` 호출을 요청한다. 응답의 `cached_tokens: 1024`는 OpenAI의 프롬프트 캐싱이다. 앞의 호출과 똑같은 앞부분(도구 설명서)이 1024토큰 이상이면 재사용해서 비용과 시간을 줄인다.

In [16]:
# 최신 정보가 필요한 질문 -> LLM이 tavily_search 호출을 요청
response = llm_with_tools.invoke("2026년 AI 트렌드는 무엇인가요?")
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 1198, 'total_tokens': 1226, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_24e955c341', 'id': 'chatcmpl-EURrqC9OxrYhKC5snB7Vll6YbedoQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fb7d-0562-7e50-b405-79b41839e981-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '2026 AI trends', 'time_range': 'year', 'topic': 'general'}, 'id': 'call_ik5MaCu977pqV35MtBAHL81Z', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1198, 'output_tokens': 28, 'total_tokens': 1226, 'input_token_details': {'audio': 0, 'cache_read': 1024}, 'output

LLM은 검색어를 그대로 넘기지 않고 `'2026 AI trends'`라는 영어 검색어로 바꿨다. 질문의 "2026년"을 보고 선택 인자 `time_range='year'`(최근 1년), `topic='general'`도 **스스로 채웠다**. 도구 설명서에 있는 인자를 LLM이 상황에 맞게 골라 쓰는 것이다.

In [17]:
# LLM이 정한 검색어와 선택 인자(time_range, topic)
response.tool_calls

[{'name': 'tavily_search',
  'args': {'query': '2026 AI trends',
   'time_range': 'year',
   'topic': 'general'},
  'id': 'call_ik5MaCu977pqV35MtBAHL81Z',
  'type': 'tool_call'}]

## 정리

| 개념 | 코드 | 핵심 |
|---|---|---|
| 타빌리 검색 도구 | `TavilySearch(max_results=2)` | 웹 검색 결과를 LLM이 쓰기 좋은 텍스트(url, title, content, score)로 돌려준다 |
| 도구 직접 실행 | `tool.invoke({"query": ...})` 또는 `tool.invoke("...")` | 인자를 넣으면 결과 딕셔너리를 돌려준다 |
| ToolCall 형식 실행 | `tool.invoke({"args":..., "type":"tool_call", "id":..., "name":...})` | 결과가 `ToolMessage`(content는 JSON 문자열, `tool_call_id` 포함)로 감싸진다 |
| 사용자 정의 도구 | `@tool` + docstring + 타입 힌트 | 함수 설명이 LLM이 읽는 도구 설명서가 된다 |
| LLM에 도구 연결 | `llm.bind_tools(tools)` | LLM은 도구를 **실행하지 않고** `tool_calls`로 실행 계획만 반환한다 |
| 도구 사용 판단 | `response.tool_calls` | 필요하면 도구 이름·인자를 채우고(여러 개 동시 가능), 필요 없으면 빈 리스트 |

- 도구 설명서도 매 호출의 입력 토큰에 포함된다. 타빌리 검색 도구 하나만 연결해도 입력 토큰이 약 1,200개가 됐다.
- LLM은 검색어를 영어로 바꾸고 `time_range` 같은 선택 인자까지 스스로 채웠다.
- 아직 LLM이 요청한 도구를 실제로 실행해서 결과를 다시 LLM에게 돌려주지는 않았다. 이 반복을 랭그래프로 만드는 게 다음 실습(6.2.3)이다.